In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from dask.distributed import wait

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
client = Client(n_workers=24,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 24
Total threads: 24,Total memory: 111.76 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43953,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46559,Total threads: 1
Dashboard: /proxy/33371/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:46373,


2025-10-16 09:31:24,966 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:45715' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('rechunk-merge-7a8409872501e672691b619b4f957fbf', 30, 0, 0), ('rechunk-merge-7a8409872501e672691b619b4f957fbf', 25, 0, 0), ('rechunk-merge-7a8409872501e672691b619b4f957fbf', 9, 0, 0), ('rechunk-merge-7a8409872501e672691b619b4f957fbf', 12, 0, 0)} (stimulus_id='handle-worker-cleanup-1760567484.9661844')
2025-10-16 09:31:25,081 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:45805' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('rechunk-merge-4c9197a3cd6e4c68672edc547caf1b72', 31, 0, 0), ('rechunk-merge-7a8409872501e672691b619b4f957fbf', 15, 0, 0), ('rechunk-merge-7a8409872501e672691b619b4f957fbf', 31, 0, 0), ('rechunk-merge-7a8409872501e672691b619b4f957fbf', 29, 0, 0), ('rechunk-merge-4c9197a3cd6e4c68672edc547caf1b72', 10, 0, 0)} (stimu

Thesse define whether which sample we're calcualting and which model we're using.

In [3]:
sample = 'baseline'
reanalysis = 'BARRA-C2'

In [4]:
if sample == 'heatwave':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")
    mode_str = 'heatwave'
    
elif sample == 'baseline':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv")
    mode_str = 'baseline'

if reanalysis == 'BARRA-C2':
    extent = [147, 152, -38.5, -32]
elif reanalysis == 'BARRA-R2':
    extent = None

lon_min, lon_max, lat_min, lat_max = extent

In [5]:
# Statistically significant w ssmin=20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'CRURWF1',
            'WOODLWN1',
            'BOCORWF1',
            'BODWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [6]:
def get_days(days, nc_dir,
             tz='Australia/Brisbane'):
    
    local_dates = pd.DatetimeIndex(pd.to_datetime(days['date'])).tz_localize(tz)
    utc_start = local_dates.min().tz_convert('UTC').normalize()
    utc_end   = (local_dates.max() + pd.Timedelta(days=1)).tz_convert('UTC').normalize()

    # Months from UTC window
    utc_months = pd.date_range(utc_start, utc_end - pd.Timedelta(seconds=1),
                               freq='MS', tz='UTC').strftime('%Y%m')
    year_month_filter = set(utc_months)

    selected_files = [
        os.path.join(nc_dir, f)
        for f in os.listdir(nc_dir)
        if any(ym in f for ym in year_month_filter)
    ]

    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks="auto"
    )

    ds_sub = ds.sel(time=slice(utc_start.tz_localize(None),
                               utc_end.tz_localize(None)))

    times_local = ds_sub.indexes['time'].tz_localize('UTC').tz_convert(tz)
    mask = times_local.normalize().isin(local_dates.normalize())
    ds_sub = ds_sub.isel(time=mask)
    ds_sub = ds_sub.assign_coords(local_hour=('time', times_local[mask].hour))
    
    return ds_sub

In [7]:
def get_ds():
    u_hw_cluster = get_days(cluster_dates, u_path)
    v_hw_cluster = get_days(cluster_dates, v_path)
    
    ds = xr.merge([u_hw_cluster, v_hw_cluster])

    ds = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
    })

    ds = ds.chunk({'time':2000,'lat':-1,'lon':-1})
    
    return ds


In [8]:
ds_subset = get_ds().persist()
wait(ds_subset)

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 171.41 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [9]:
ds_subset

<xarray.Dataset> Size: 25GB
Dimensions:     (time: 77497, lat: 163, lon: 125)
Coordinates:
  * time        (time) datetime64[ns] 620kB 2014-09-01 ... 2024-06-30
    height      float64 8B 100.0
  * lon         (lon) float64 1kB 147.0 147.1 147.1 147.1 ... 151.9 151.9 152.0
  * lat         (lat) float64 1kB -38.49 -38.45 -38.41 ... -32.09 -32.05 -32.01
    crs         int32 4B 0
    local_hour  (time) int32 310kB dask.array<chunksize=(2000,), meta=np.ndarray>
Data variables:
    ua100m      (time, lat, lon) float64 13GB dask.array<chunksize=(2000, 163, 125), meta=np.ndarray>
    va100m      (time, lat, lon) float64 13GB dask.array<chunksize=(2000, 163, 125), meta=np.ndarray>
Attributes: (12/60)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        500df2a
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T00:33:05Z
    date_metadata_modified:    2024-10-11T00:33:05Z
    history:                   Wed Jun 12 13:46:27 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [10]:
# This computes the composite of variance
ds_subset['windspeed'] = np.sqrt(ds_subset['ua100m']**2 + ds_subset['va100m']**2)

def temporal_variance(x):
    # x has dimensions (time, lat, lon)
    return x.var(dim="time", ddof=1)  # use ddof=1 for unbiased sample variance

hourly_var_per_point = ds_subset['windspeed'].groupby("time.hour").map(temporal_variance)

In [11]:
# This computes the hourly u v field composite.
hourly_composite =  ds_subset.groupby("time.hour").mean()

In [12]:
# This computes the composites of minimum, maximum, and diunal amplitude
def compute_windspeed_composites(windspeed):
    """
    Compute windspeed composites (max, min, amplitude) from u and v.
    
    Parameters
    ----------
    u, v : xarray.DataArray
        Wind vector components (time x lat x lon)
    
    Returns
    -------
    dict
        {"max": DataArray, "min": DataArray, "diff": DataArray}
    """
    # Compute windspeed
    
    # Compute composites along time dimension
    composite_max = windspeed.max(dim="time")
    composite_min = windspeed.min(dim="time")
    diurnal_amp = (composite_max - composite_min)
    
    return composite_max, composite_min, diurnal_amp

max_speed, min_speed, diurnal_amp = compute_windspeed_composites(ds_subset['windspeed'])

In [13]:
ds_subset = ds_subset.assign_coords(local_hour=("time", ds_subset["local_hour"].values))

hourly_var_per_point = ds_subset['windspeed'].groupby("local_hour").map(temporal_variance)
hourly_composite = ds_subset.groupby("local_hour").mean()

results = xr.Dataset(
    {
        "windspeed_variance": hourly_var_per_point,
        "windspeed_mean": hourly_composite["windspeed"],
        "u_mean": hourly_composite["ua100m"],
        "v_mean": hourly_composite["va100m"],
        "windspeed_max": max_speed,
        "windspeed_min": min_speed,
        "diurnal_amp": diurnal_amp,
    }
)

results = results.rename({"local_hour": "hour"})
results = results.assign_attrs(reanalysis=reanalysis, mode=mode_str)

# Write to NetCDF lazily with dask
with ProgressBar():
    results.to_netcdf(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc", compute=True)

2025-10-16 09:16:40,201 - distributed.worker.memory - WARNING - gc.collect() took 1.063s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.


In [14]:
xr.open_dataset(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc")

<xarray.Dataset> Size: 16MB
Dimensions:             (lon: 125, lat: 163, hour: 24)
Coordinates:
    height              float64 8B ...
  * lon                 (lon) float64 1kB 147.0 147.1 147.1 ... 151.9 152.0
  * lat                 (lat) float64 1kB -38.49 -38.45 -38.41 ... -32.05 -32.01
    crs                 int32 4B ...
  * hour                (hour) int32 96B 0 1 2 3 4 5 6 ... 17 18 19 20 21 22 23
Data variables:
    windspeed_variance  (hour, lat, lon) float64 4MB ...
    windspeed_mean      (hour, lat, lon) float64 4MB ...
    u_mean              (hour, lat, lon) float64 4MB ...
    v_mean              (hour, lat, lon) float64 4MB ...
    windspeed_max       (lat, lon) float64 163kB ...
    windspeed_min       (lat, lon) float64 163kB ...
    diurnal_amp         (lat, lon) float64 163kB ...
Attributes:
    reanalysis:  BARRA-C2
    mode:        baseline